# Compress Intan recordings with WavPack

Use this to archive standard headerless Intan `.dat` streams as smaller `.wv` files.
Compression is **lossless**: restoring an archive recreates the original file's
bytes. These functions support little-endian integer samples (`int16`, `uint16`,
`int32`, or `uint32`) with 1–4096 saved channels. Multichannel data must be
interleaved by sample, as in Intan's "One File Per Signal Type" format.

Run the cells below in order after changing the settings for your recording.

## 1. Install the tools

You need the **WavPack command-line tools**; installing the Python package
does not install them. Choose your operating system:

- **Windows:** download the 64-bit command-line ZIP from the
  [official WavPack downloads page](https://www.wavpack.com/downloads.html), then
  extract it. Add the folder containing `wavpack.exe` and `wvunpack.exe` to your
  Windows `PATH`, then restart your terminal or notebook. Alternatively, enter
  the full executable paths in the settings below.
- **macOS:** run `brew install wavpack` in Terminal, using
  [Homebrew](https://brew.sh/).
- **Ubuntu/Debian Linux:** run `sudo apt install wavpack` in a terminal.

Run `wavpack --version` and `wvunpack --version` in your terminal to check that
both tools are available. **For recordings with more than 256 saved channels, use
WavPack 5.6 or newer.** If your Linux package manager provides an older version,
install a current release from the
[official source downloads](https://www.wavpack.com/downloads.html).

In [ ]:
from pathlib import Path

from neuro_py.raw import compress_dat, decompress_dat

## 2. Set your recording details

Change `session_dir` and `n_channels` below. The channel count must match the
channels **actually saved in `amplifier.dat`**; `256` is only an example.
`sample_rate=20_000` is our standard 20 kHz recording rate. Check `info.rhd` if
unsure, and change the rate if your session differs.

On macOS or Linux, use a path such as `Path("/data/session1")`. Leave the two
executable settings as `None` when the tools are on `PATH`.

In [ ]:
session_dir = Path(r"D:\recordings\session1")
n_channels = 256  # Replace with the amplifier channels actually saved
sample_rate = 20_000  # Hz

wavpack_executable = None  # Or r"C:\Tools\WavPack\wavpack.exe"
wvunpack_executable = None  # Or r"C:\Tools\WavPack\wvunpack.exe"

## 3. Compress the amplifier recording

The default `dtype="int16"` matches standard Intan amplifier data. Pass the
original integer recording directly, without converting it to microvolts.

The original `.dat` stays in place. Verification is enabled by default: WavPack
stores a checksum and fully reads and decodes the archive to check the samples.
This extra pass takes time. Keep enough free space for the original and archive.

In [ ]:
archive = compress_dat(
    input_path=session_dir / "amplifier.dat",
    n_channels=n_channels,
    sample_rate=sample_rate,
    executable=wavpack_executable,
)
print(f"Archive: {archive}")

## 4. Restore the recording

Restore to a different filename while the original is still present. Restoration
also verifies the stored checksum by default. You do not need to supply the
channel count, sample rate, or `dtype` again.

In [ ]:
restored = decompress_dat(
    input_path=archive,
    output_path=session_dir / "amplifier_restored.dat",
    executable=wvunpack_executable,
)
print(f"Restored recording: {restored}")

Existing output files are never overwritten. Rerunning these cells with the same
filenames therefore raises `FileExistsError`. To make another archive or restored
copy, choose a new `output_path`. If you omit `output_path` during restoration,
the default is `amplifier.dat`, which fails safely when the original exists.

## 5. Optional: archive other Intan streams

Choose `dtype` for the type of file you are compressing:

| Intan file | `dtype` | `n_channels` |
| --- | --- | --- |
| `amplifier.dat` | `"int16"` (default) | Saved amplifier channel count |
| `time.dat` | `"int32"` | `1` |
| `digitalin.dat`, `digitalout.dat` | `"uint16"` | `1` (all TTL lines share one packed word) |
| `auxiliary.dat`, `supply.dat`, `analogin.dat` | `"uint16"` | Saved channel count for that file |

These settings follow the
[Intan RHD file format](https://intantech.com/files/Intan_RHD2000_data_file_formats.pdf).
For these standard files, use the amplifier sample rate: auxiliary and supply
samples are repeated in the `.dat` files to align with `time.dat`. Compress the
original integer files directly, without converting values to volts or seconds.
If another tool has resampled a file, use that file's actual stored rate instead.

The optional cell below compresses timestamps and packed digital streams when
they exist, and prints a skip message for missing files. Each file gets its own
archive. For auxiliary, supply, or analog input files, use the same call with
`dtype="uint16"` and that file's actual saved channel count.

In [ ]:
for filename, dtype in [
    ("time.dat", "int32"),
    ("digitalin.dat", "uint16"),
    ("digitalout.dat", "uint16"),
]:
    source = session_dir / filename
    if source.is_file():
        stream_archive = compress_dat(
            input_path=source,
            n_channels=1,
            sample_rate=sample_rate,
            dtype=dtype,
            executable=wavpack_executable,
        )
        print(f"Archive: {stream_archive}")
    else:
        print(f"Skipped {filename}: file is not present in this session.")

Unsigned values and packed digital bits are preserved byte for byte. Read
restored files using their original Intan `dtype`; WavPack's internal PCM metadata
uses signed integers even for unsigned input.

For more than 256 channels, pass the real saved channel count as usual;
the function handles it automatically.

To restore without Python, use WavPack's raw-output mode in a terminal:

```bash
wvunpack -r -m amplifier.wv -o amplifier_restored.dat
```

## What to keep

Keep every `.wv` archive **and the rest of the recording folder**: `info.rhd`,
tracking, behavior, and other session metadata. Keep any `.dat` streams you
have not compressed. Each archive contains only the file you passed to it, not
the rest of the session. Keep a copy of these instructions and the WavPack tools
with your archives. Before removing an original recording yourself, restore and
check your first archive and verify the copies on your destination drive.

For the function options, see the
[WavPack API reference](../reference/neuro_py/raw/wavpack.md).